In [41]:
from google.colab import files
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

In [2]:
upload = files.upload()

Saving battery_synthetic_dataset.csv to battery_synthetic_dataset.csv


In [42]:
df = pd.read_csv("battery_synthetic_dataset.csv")

In [43]:
data = df.copy()

x = data.drop("label", axis=1)
y = data["label"]

In [44]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [45]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [46]:
input_size = x_train.shape[1]
hidden1_size = 5
hidden2_size = 7
hidden3_size = 10
hidden4_size = 5
output_size = 1

In [47]:
def sigmoid(x):
    return 1/(1+np.exp(-x))
def sigmoid_derivative(x):
    return x*(1-x)
def relu(x):
    return np.maximum(0, x)
def relu_derivative(x):
    return (x > 0).astype(float)

In [48]:
rng = np.random.default_rng(42)

def he_init(fan_in, fan_out):
    return rng.standard_normal((fan_in, fan_out)) * np.sqrt(2/fan_in)

In [49]:
weight_input_hidden1 = he_init(input_size, hidden1_size)
bias_input_hidden1 = np.zeros((1, hidden1_size))

weight_hidden1_hidden2 = he_init(hidden1_size, hidden2_size)
bias_hidden1_hidden2 = np.zeros((1, hidden2_size))

weight_hidden2_hidden3 = he_init(hidden2_size, hidden3_size)
bias_hidden2_hidden3 = np.zeros((1, hidden3_size))

weight_hidden3_hidden4 = he_init(hidden3_size, hidden4_size)
bias_hidden3_hidden4 = np.zeros((1, hidden4_size))

weight_hidden4_output = he_init(hidden4_size, output_size)
bias_hidden4_output = np.zeros((1, output_size))

In [50]:
epochs = 10000
learning_rate = 1e-2
n = x_train.shape[0]

In [51]:
for epoch in range(epochs):
    input = x_train

    hidden1_input = np.dot(input, weight_input_hidden1) + bias_input_hidden1
    hidden1_output = relu(hidden1_input)

    hidden2_input = np.dot(hidden1_output, weight_hidden1_hidden2) + bias_hidden1_hidden2
    hidden2_output = relu(hidden2_input)

    hidden3_input = np.dot(hidden2_output, weight_hidden2_hidden3) + bias_hidden2_hidden3
    hidden3_output = relu(hidden3_input)

    hidden4_input = np.dot(hidden3_output, weight_hidden3_hidden4) + bias_hidden3_hidden4
    hidden4_output = relu(hidden4_input)

    output_input = np.dot(hidden4_output, weight_hidden4_output) + bias_hidden4_output
    output = sigmoid(output_input)

    # backward pass
    error = y_train.values.reshape(-1, 1) - output
    delta_output = error * sigmoid_derivative(output)
    delta_hidden4 = delta_output.dot(weight_hidden4_output.T) * relu_derivative(hidden4_input)
    delta_hidden3 = delta_hidden4.dot(weight_hidden3_hidden4.T) * relu_derivative(hidden3_input)
    delta_hidden2 = delta_hidden3.dot(weight_hidden2_hidden3.T) * relu_derivative(hidden2_input)
    delta_hidden1 = delta_hidden2.dot(weight_hidden1_hidden2.T) * relu_derivative(hidden1_input)

    gradient_hidden4_output = hidden4_output.T.dot(delta_output) / n
    gradient_hidden3_hidden4 = hidden3_output.T.dot(delta_hidden4) / n
    gradient_hidden2_hidden3 = hidden2_output.T.dot(delta_hidden3) / n
    gradient_hidden1_hidden2 = hidden1_output.T.dot(delta_hidden2) / n
    gradient_input_hidden1 = input.T.dot(delta_hidden1) / n

    bias_hidden4_output += learning_rate * np.mean(delta_output, axis=0, keepdims=True)
    bias_hidden3_hidden4 += learning_rate * np.mean(delta_hidden4, axis=0, keepdims=True)
    bias_hidden2_hidden3 += learning_rate * np.mean(delta_hidden3, axis=0, keepdims=True)
    bias_hidden1_hidden2 += learning_rate * np.mean(delta_hidden2, axis=0, keepdims=True)
    bias_input_hidden1 += learning_rate * np.mean(delta_hidden1, axis=0, keepdims=True)

    weight_hidden4_output += learning_rate * gradient_hidden4_output
    weight_hidden3_hidden4 += learning_rate * gradient_hidden3_hidden4
    weight_hidden2_hidden3 += learning_rate * gradient_hidden2_hidden3
    weight_hidden1_hidden2 += learning_rate * gradient_hidden1_hidden2
    weight_input_hidden1 += learning_rate * gradient_input_hidden1

    if (epoch+1) % 1000 == 0 or epoch == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss}")

    gradiant.append((gradient_input_hidden1,gradient_hidden1_hidden2,
                     gradient_hidden2_hidden3,gradient_hidden3_hidden4,
                     gradient_hidden4_output))

Epoch 1/10000 - Loss: 0.26063795953987673
Epoch 1000/10000 - Loss: 0.1509669778319373
Epoch 2000/10000 - Loss: 0.08410587376280883
Epoch 3000/10000 - Loss: 0.07575580247688304
Epoch 4000/10000 - Loss: 0.07245126320484249
Epoch 5000/10000 - Loss: 0.07052527090149224
Epoch 6000/10000 - Loss: 0.06921254073253236
Epoch 7000/10000 - Loss: 0.06827617535053042
Epoch 8000/10000 - Loss: 0.06756845679065508
Epoch 9000/10000 - Loss: 0.06700771820948116
Epoch 10000/10000 - Loss: 0.06655922809325057


In [52]:
hidden1_input = np.dot(x_test,weight_input_hidden1) + bias_input_hidden1
hidden1_output = relu(hidden1_input)

hidden2_input = np.dot(hidden1_output,weight_hidden1_hidden2) + bias_hidden1_hidden2
hidden2_output = relu(hidden2_input)

hidden3_input = np.dot(hidden2_output,weight_hidden2_hidden3) + bias_hidden2_hidden3
hidden3_output = relu(hidden3_input)

hidden4_input = np.dot(hidden3_output,weight_hidden3_hidden4) + bias_hidden3_hidden4
hidden4_output = relu(hidden4_input)

output_input = np.dot(hidden4_output,weight_hidden4_output) + bias_hidden4_output
output = sigmoid(output_input)

In [53]:
predicted_labels = (output > 0.5).astype(int)
print(classification_report(y_test, predicted_labels))

              precision    recall  f1-score   support

           0       0.85      0.92      0.88        73
           1       0.95      0.91      0.93       127

    accuracy                           0.91       200
   macro avg       0.90      0.91      0.90       200
weighted avg       0.91      0.91      0.91       200



# Feature Importance

## Permutation Importance

In [57]:
def predict(x, weights_biases):
    (weight_input_hidden1, bias_input_hidden1,
     weight_hidden1_hidden2, bias_hidden1_hidden2,
     weight_hidden2_hidden3, bias_hidden2_hidden3,
     weight_hidden3_hidden4, bias_hidden3_hidden4,
     weight_hidden4_output, bias_hidden4_output) = weights_biases

    h1 = relu(np.dot(x, weight_input_hidden1) + bias_input_hidden1)
    h2 = relu(np.dot(h1, weight_hidden1_hidden2) + bias_hidden1_hidden2)
    h3 = relu(np.dot(h2, weight_hidden2_hidden3) + bias_hidden2_hidden3)
    h4 = relu(np.dot(h3, weight_hidden3_hidden4) + bias_hidden3_hidden4)
    out = sigmoid(np.dot(h4, weight_hidden4_output) + bias_hidden4_output)
    return out


def permutation_importance(x_test, y_test, weights_biases, feature_names, n_repeats=10, seed=42):
    rng = np.random.default_rng(seed)

    # baseline accuracy
    baseline_pred = (predict(x_test, weights_biases) > 0.7).astype(int)
    baseline_acc = accuracy_score(y_test, baseline_pred)

    importances = {}

    for col_idx, feature_name in enumerate(feature_names):
        drops = []
        for _ in range(n_repeats):
            x_permuted = x_test.copy()
            x_permuted[:, col_idx] = rng.permutation(x_permuted[:, col_idx])

            permuted_pred = (predict(x_permuted, weights_biases) > 0.7).astype(int)
            permuted_acc = accuracy_score(y_test, permuted_pred)

            drops.append(baseline_acc - permuted_acc)

        importances[feature_name] = {
            "mean_drop": np.mean(drops),
            "std_drop": np.std(drops)
        }

    return baseline_acc, importances


weights_biases = (
    weight_input_hidden1, bias_input_hidden1,
    weight_hidden1_hidden2, bias_hidden1_hidden2,
    weight_hidden2_hidden3, bias_hidden2_hidden3,
    weight_hidden3_hidden4, bias_hidden3_hidden4,
    weight_hidden4_output, bias_hidden4_output
)

feature_names = x.columns.tolist()

baseline_acc, importances = permutation_importance(x_test, y_test, weights_biases, feature_names)

print(f"Baseline Accuracy: {baseline_acc:.4f}\n")

sorted_importances = sorted(importances.items(), key=lambda item: item[1]["mean_drop"], reverse=True)
for name, stats in sorted_importances:
    print(f"{name}: {stats['mean_drop']:.4f} (± {stats['std_drop']:.4f})")

Baseline Accuracy: 0.8900

avg_voltage: 0.0925 (± 0.0242)
avg_temperature: 0.0425 (± 0.0310)
battery_life_months: 0.0305 (± 0.0131)
cycle_count: 0.0135 (± 0.0161)


##  Integrated Gradients

In [59]:
def forward_with_cache(x_input, weights_biases):
    (weight_input_hidden1, bias_input_hidden1,
     weight_hidden1_hidden2, bias_hidden1_hidden2,
     weight_hidden2_hidden3, bias_hidden2_hidden3,
     weight_hidden3_hidden4, bias_hidden3_hidden4,
     weight_hidden4_output, bias_hidden4_output) = weights_biases

    h1_in = np.dot(x_input, weight_input_hidden1) + bias_input_hidden1
    h1_out = relu(h1_in)

    h2_in = np.dot(h1_out, weight_hidden1_hidden2) + bias_hidden1_hidden2
    h2_out = relu(h2_in)

    h3_in = np.dot(h2_out, weight_hidden2_hidden3) + bias_hidden2_hidden3
    h3_out = relu(h3_in)

    h4_in = np.dot(h3_out, weight_hidden3_hidden4) + bias_hidden3_hidden4
    h4_out = relu(h4_in)

    out_in = np.dot(h4_out, weight_hidden4_output) + bias_hidden4_output
    out = sigmoid(out_in)

    cache = (h1_in, h1_out, h2_in, h2_out, h3_in, h3_out, h4_in, h4_out, out_in, out)
    return out, cache


def analytical_gradient(x_input, weights_biases):
    (weight_input_hidden1, bias_input_hidden1,
     weight_hidden1_hidden2, bias_hidden1_hidden2,
     weight_hidden2_hidden3, bias_hidden2_hidden3,
     weight_hidden3_hidden4, bias_hidden3_hidden4,
     weight_hidden4_output, bias_hidden4_output) = weights_biases

    output, cache = forward_with_cache(x_input, weights_biases)
    (h1_in, h1_out, h2_in, h2_out, h3_in, h3_out, h4_in, h4_out, out_in, out) = cache

    d_out = np.ones_like(out)

    delta_out = d_out * sigmoid_derivative(out)

    delta_h4 = delta_out.dot(weight_hidden4_output.T) * relu_derivative(h4_in)
    delta_h3 = delta_h4.dot(weight_hidden3_hidden4.T) * relu_derivative(h3_in)
    delta_h2 = delta_h3.dot(weight_hidden2_hidden3.T) * relu_derivative(h2_in)
    delta_h1 = delta_h2.dot(weight_hidden1_hidden2.T) * relu_derivative(h1_in)

    d_input = delta_h1.dot(weight_input_hidden1.T)

    return d_input


def integrated_gradients_analytical(x_sample, weights_biases, baseline=None, m_steps=50):
    if baseline is None:
        baseline = np.zeros_like(x_sample)

    alphas = np.linspace(0, 1, m_steps)

    total_gradients = np.zeros_like(x_sample)

    for alpha in alphas:
        interpolated_x = baseline + alpha * (x_sample - baseline)
        grad = analytical_gradient(interpolated_x, weights_biases)
        total_gradients += grad

    avg_gradients = total_gradients / m_steps

    integrated_grads = (x_sample - baseline) * avg_gradients

    return integrated_grads.flatten()


sample = x_test[0:1]
ig_values = integrated_gradients_analytical(sample, weights_biases, m_steps=50)

for name, val in sorted(zip(feature_names, ig_values), key=lambda p: abs(p[1]), reverse=True):
    print(f"{name}: {val:.5f}")

baseline = np.zeros_like(sample)
output_sample, _ = forward_with_cache(sample, weights_biases)
output_baseline, _ = forward_with_cache(baseline, weights_biases)

print(f"\nIG: {ig_values.sum():.5f}")
print(f"F(x) - F(baseline): {(output_sample - output_baseline).item():.5f}")

cycle_count: 0.05012
avg_voltage: 0.02762
avg_temperature: 0.01668
battery_life_months: -0.00386

IG: 0.09055
F(x) - F(baseline): 0.08859


In [64]:
all_ig = []
for i in range(x_test.shape[0]):
    sample = x_test[i:i+1]
    ig_values = integrated_gradients_analytical(sample, weights_biases, m_steps=50)
    all_ig.append(ig_values)

all_ig = np.array(all_ig)
mean_abs_ig = np.mean(np.abs(all_ig), axis=0)

for name, val in sorted(zip(feature_names, mean_abs_ig), key=lambda p: p[1], reverse=True):
    print(f"{name}: {val:.5f}")

avg_voltage: 0.13084
battery_life_months: 0.09864
cycle_count: 0.08087
avg_temperature: 0.07079
